In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

# Load historical fights (adjust path if running from a subfolder)
df = pd.read_csv("../build/fight_ml_dataset.csv")

# Pick the binary target column (1 if fighter A wins)
candidates = ['A_wins','winner_A','is_A_winner','target','label','result']
y_col = next((c for c in candidates if c in df.columns), None)
if y_col is None:
    # Fallback: auto-detect a binary column that looks like a result
    for c in df.columns:
        vals = pd.Series(df[c]).dropna().unique()
        if set(vals).issubset({0,1}) and ('win' in c.lower() or 'result' in c.lower()):
            y_col = c
            break
if y_col is None:
    raise ValueError("Could not find a binary target column. Set y_col to your label column name.")

# Keep rows with ELO and label
mask = df['current_elo_rating_A'].notna() & df['current_elo_rating_B'].notna() & df[y_col].notna()
d = df.loc[mask].copy()

# Elo expected probability of fighter A
pA = 1.0 / (1.0 + 10.0 ** ((d['current_elo_rating_B'] - d['current_elo_rating_A']) / 400.0))
pred = (pA >= 0.5).astype(int)
y = d[y_col].astype(int)

# Metrics
acc = accuracy_score(y, pred)
auc = roc_auc_score(y, pA)
cm = confusion_matrix(y, pred)

print(f"Rows used: {len(d)}")
print(f"Elo-only accuracy: {acc:.3f}")
print(f"Elo-only ROC AUC: {auc:.3f}")
print("Confusion matrix (y=[0,1]):\n", cm)
print("\nReport:\n", classification_report(y, pred, digits=3))

Rows used: 8138
Elo-only accuracy: 0.771
Elo-only ROC AUC: 0.830
Confusion matrix (y=[0,1]):
 [[2053  834]
 [1031 4220]]

Report:
               precision    recall  f1-score   support

           0      0.666     0.711     0.688      2887
           1      0.835     0.804     0.819      5251

    accuracy                          0.771      8138
   macro avg      0.750     0.757     0.753      8138
weighted avg      0.775     0.771     0.772      8138

